In [1]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)   # empty slices where a whole row was dropped
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

DATA = '../data'
X_train = np.load(f'{DATA}/train_cadences.npy', mmap_mode='r')   # (18000, 6, 16, 128) uint8
X_test  = np.load(f'{DATA}/test_cadences.npy', mmap_mode='r')    # (10000, 6, 16, 128) uint8
train = pd.read_csv(f'{DATA}/train_meta.csv')
test  = pd.read_csv(f'{DATA}/test_meta.csv')
print(X_train.shape, X_test.shape)
print(train.label.value_counts().to_dict())

(18000, 6, 16, 128) (10000, 6, 16, 128)
{'noise': 4211, 'rfi_persistent': 2892, 'rfi_intermittent': 2662, 'radar_chirp': 1849, 'satellite_pass': 1805, 'pulsar': 1629, 'scintillating_star': 1431, 'frb': 1042, 'eti_candidate': 479}


In [6]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report

# Load the training labels using your existing project path.
# train = pd.read_pickle("../data/train.pkl")  # Adjust if your labels use a different file.

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(train["label"])

n_classes = len(label_encoder.classes_)

print("Number of classes:", n_classes)
print("Classes:", list(label_encoder.classes_))

Number of classes: 9
Classes: ['eti_candidate', 'frb', 'noise', 'pulsar', 'radar_chirp', 'rfi_intermittent', 'rfi_persistent', 'satellite_pass', 'scintillating_star']


In [7]:

import torch
import torch.nn as nn

class SmallSignalCNN(nn.Module):
    def __init__(self, n_classes):
        super().__init__()

        self.features = nn.Sequential(
            # Input: (batch, 12, 16, 128)
            nn.Conv2d(12, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # (16, 8, 64)

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # (32, 4, 32)

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            # Compact representation
            nn.AdaptiveAvgPool2d((1, 4))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),             # 64 * 1 * 4 = 256
            nn.Linear(256, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(64, n_classes)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


In [8]:
# Get the number of classes from the label encoder created in E5.
n_classes = len(label_encoder.classes_)

print("Number of classes:", n_classes)
print("Classes:", list(label_encoder.classes_))

Number of classes: 9
Classes: ['eti_candidate', 'frb', 'noise', 'pulsar', 'radar_chirp', 'rfi_intermittent', 'rfi_persistent', 'satellite_pass', 'scintillating_star']


In [10]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Selected device:", device)

Selected device: cpu


In [12]:
import numpy as np
import torch

# Count examples belonging to each class in the full training set.
class_counts = np.bincount(
    y_encoded,
    minlength=n_classes
)

# Same weighting strategy used in E5.
class_weights = 1.0 / np.sqrt(class_counts)

# Normalize weights to have a mean of 1.
class_weights = class_weights / class_weights.mean()

# Convert to a PyTorch tensor.
class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32
)

print("Class counts:", class_counts)
print("Class weights:", class_weights)

Class counts: [ 479 1042 4211 1629 1849 2662 2892 1805 1431]
Class weights: tensor([1.8031, 1.2225, 0.6081, 0.9778, 0.9177, 0.7649, 0.7338, 0.9289, 1.0432])


In [13]:

# Reproducibility
import random
import numpy as np
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

model_cnn = SmallSignalCNN(n_classes=n_classes).to(device)

criterion = nn.CrossEntropyLoss(
    weight=class_weights.to(device)
)

optimizer = torch.optim.AdamW(
    model_cnn.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

num_params = sum(
    p.numel() for p in model_cnn.parameters() if p.requires_grad
)

print(f"Device: {device}")
print(f"Trainable parameters: {num_params:,}")


Device: cpu
Trainable parameters: 42,137


In [14]:
from sklearn.model_selection import train_test_split
import numpy as np

indices = np.arange(len(y_encoded))

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.2,
    stratify=y_encoded,
    random_state=42
)

print("Training samples:", len(train_idx))
print("Validation samples:", len(val_idx))
print("Total samples:", len(indices))

Training samples: 14400
Validation samples: 3600
Total samples: 18000


In [15]:
required_vars = [
    "X_train",
    "y_encoded",
    "train_idx",
    "val_idx",
    "prepare_cnn_input",
    "SignalDataset",
    "device",
    "class_weights",
]

for name in required_vars:
    print(f"{name}: {'Available' if name in globals() else 'MISSING'}")

X_train: Available
y_encoded: Available
train_idx: Available
val_idx: Available
prepare_cnn_input: MISSING
SignalDataset: MISSING
device: Available
class_weights: Available


In [16]:


from torch.utils.data import Dataset, DataLoader

def prepare_cnn_input(X):
    """
    Input shape: (N, 6, 16, 128)
    Output shape: (N, 12, 16, 128)

    First 6 channels: normalized signal
    Next 6 channels: missing-pixel masks
    """
    X = X.astype(np.float32)

    # Zero means missing.
    mask = (X != 0).astype(np.float32)

    # Calculate mean/std over observed pixels per sample and panel.
    observed = mask.sum(axis=(2, 3), keepdims=True)
    observed = np.maximum(observed, 1.0)

    values_sum = (X * mask).sum(axis=(2, 3), keepdims=True)
    mean = values_sum / observed

    centered = (X - mean) * mask
    variance = (centered ** 2).sum(
        axis=(2, 3), keepdims=True
    ) / observed

    std = np.sqrt(variance)
    std = np.maximum(std, 1e-6)

    normalized = centered / std
    normalized *= mask

    return np.concatenate([normalized, mask], axis=1)


class SignalDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.as_tensor(X, dtype=torch.float32)
        self.y = torch.as_tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


In [17]:

# Prepare only the samples needed for each split.
X_train_cnn = prepare_cnn_input(X_train[train_idx])
X_val_cnn = prepare_cnn_input(X_train[val_idx])

y_train_split = y_encoded[train_idx]
y_val_split = y_encoded[val_idx]

train_dataset = SignalDataset(X_train_cnn, y_train_split)
val_dataset = SignalDataset(X_val_cnn, y_val_split)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=0
)

print("Train input:", X_train_cnn.shape)
print("Validation input:", X_val_cnn.shape)
print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))


Train input: (14400, 12, 16, 128)
Validation input: (3600, 12, 16, 128)
Train batches: 225
Validation batches: 29


In [18]:

import copy
import torch
import numpy as np
from sklearn.metrics import f1_score

MAX_EPOCHS = 20
PATIENCE = 4

best_val_f1 = -1.0
best_epoch = 0
best_state = None
epochs_without_improvement = 0

for epoch in range(1, MAX_EPOCHS + 1):
    # Training
    model_cnn.train()
    train_loss_sum = 0.0
    train_samples = 0

    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        logits = model_cnn(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        batch_size = yb.size(0)
        train_loss_sum += loss.item() * batch_size
        train_samples += batch_size

    train_loss = train_loss_sum / train_samples

    # Validation
    model_cnn.eval()
    val_preds = []
    val_targets = []

    with torch.no_grad():
        for xb, yb in val_loader:
            logits = model_cnn(xb.to(device))
            preds = logits.argmax(dim=1)

            val_preds.extend(preds.cpu().numpy())
            val_targets.extend(yb.numpy())

    val_f1 = f1_score(
        val_targets,
        val_preds,
        average="macro",
        zero_division=0
    )

    print(
        f"Epoch {epoch:02d} | "
        f"loss={train_loss:.4f} | "
        f"val_macro_f1={val_f1:.4f}"
    )

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_epoch = epoch
        best_state = {
            k: v.detach().cpu().clone()
            for k, v in model_cnn.state_dict().items()
        }
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:
        print("Early stopping.")
        break

# Restore the best validation checkpoint.
if best_state is None:
    raise RuntimeError("Training did not produce a valid checkpoint.")

model_cnn.load_state_dict(best_state)
model_cnn.to(device)

print(f"\nBest epoch: {best_epoch}")
print(f"Best validation macro-F1: {best_val_f1:.4f}")

torch.save(
    best_state,
    "../outputs/features/e6_small_cnn_best.pt"
)


Epoch 01 | loss=1.6362 | val_macro_f1=0.1537
Epoch 02 | loss=1.2725 | val_macro_f1=0.2462
Epoch 03 | loss=1.1365 | val_macro_f1=0.4517
Epoch 04 | loss=1.0597 | val_macro_f1=0.1206
Epoch 05 | loss=1.0062 | val_macro_f1=0.3597
Epoch 06 | loss=0.9640 | val_macro_f1=0.1053
Epoch 07 | loss=0.9309 | val_macro_f1=0.4919
Epoch 08 | loss=0.9024 | val_macro_f1=0.4550
Epoch 09 | loss=0.8755 | val_macro_f1=0.4581
Epoch 10 | loss=0.8503 | val_macro_f1=0.4735
Epoch 11 | loss=0.8353 | val_macro_f1=0.5356
Epoch 12 | loss=0.8099 | val_macro_f1=0.3276
Epoch 13 | loss=0.7989 | val_macro_f1=0.5590
Epoch 14 | loss=0.7801 | val_macro_f1=0.5720
Epoch 15 | loss=0.7555 | val_macro_f1=0.4830
Epoch 16 | loss=0.7447 | val_macro_f1=0.5652
Epoch 17 | loss=0.7321 | val_macro_f1=0.4418
Epoch 18 | loss=0.7134 | val_macro_f1=0.3218
Early stopping.

Best epoch: 14
Best validation macro-F1: 0.5720


In [19]:
from sklearn.metrics import f1_score
import torch

model_cnn.eval()

train_preds = []
train_targets = []

# Use a non-shuffled loader over the training dataset.
train_eval_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=0
)

with torch.no_grad():
    for xb, yb in train_eval_loader:
        logits = model_cnn(xb.to(device))
        preds = logits.argmax(dim=1)

        train_preds.extend(preds.cpu().numpy())
        train_targets.extend(yb.numpy())

e6_train_f1 = f1_score(
    train_targets,
    train_preds,
    average="macro",
    zero_division=0
)

print(f"E6 training macro-F1:   {e6_train_f1:.4f}")
print(f"E6 validation macro-F1: {best_val_f1:.4f}")
print(f"Train-validation gap:   {e6_train_f1 - best_val_f1:.4f}")

E6 training macro-F1:   0.6127
E6 validation macro-F1: 0.5720
Train-validation gap:   0.0406
